# Silver Layer — Bronze to Silver

Reads `bronze_nyc311_new`, casts it to strict types, keeps one row per service
request, and upserts into `silver_nyc311_new` using `MERGE INTO`.

**Parameters**

| `silver_mode` | What it does |
|---|---|
| `batch` | Processes the one batch named in `batch_id` |
| `backfill_all` | Re-processes every batch in Bronze |
| `full_rebuild` | Drops Silver, then rebuilds it from all of Bronze |

Silver is entirely derived from Bronze, so nothing is lost by rebuilding. Any
mode can be re-run any number of times without creating duplicates.

## 1. Parameters

In [ ]:
from pyspark.sql.functions import (col, trim, coalesce, current_timestamp,
                                   row_number, when, lit)
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               TimestampType, IntegerType, DoubleType)
from delta.tables import DeltaTable
from datetime import datetime

dbutils.widgets.text("batch_id", "page_3")
dbutils.widgets.dropdown("silver_mode", "batch",
                         ["batch", "backfill_all", "full_rebuild"])

batch_id    = dbutils.widgets.get("batch_id").strip()
silver_mode = dbutils.widgets.get("silver_mode")

BRONZE     = "workspace.default.bronze_nyc311_new"
SILVER     = "workspace.default.silver_nyc311_new"
QUARANTINE = "workspace.default.silver_quarantine_new"
LOG_TABLE  = "workspace.default.pipeline_execution_logs_new"

print("mode:", silver_mode, "| batch:", batch_id)

## 2. Silver schema (explicit contract)

Defined with `StructType` and `StructField`, never inferred. This is the
contract every row must satisfy before it is allowed into Silver.

In [ ]:
silver_schema = StructType([
  StructField("unique_key",                     StringType(),    False),
  StructField("created_date",                   TimestampType(), True),
  StructField("closed_date",                    TimestampType(), True),
  StructField("due_date",                       TimestampType(), True),
  StructField("resolution_action_updated_date", TimestampType(), True),
  StructField("agency",                         StringType(),    True),
  StructField("agency_name",                    StringType(),    True),
  StructField("complaint_type",                 StringType(),    True),
  StructField("descriptor",                     StringType(),    True),
  StructField("location_type",                  StringType(),    True),
  StructField("status",                         StringType(),    True),
  StructField("resolution_description",         StringType(),    True),
  StructField("incident_zip",                   StringType(),    True),
  StructField("city",                           StringType(),    True),
  StructField("borough",                        StringType(),    True),
  StructField("community_board",                StringType(),    True),
  StructField("council_district",               IntegerType(),   True),
  StructField("police_precinct",                StringType(),    True),
  StructField("open_data_channel_type",         StringType(),    True),
  StructField("address_type",                   StringType(),    True),
  StructField("latitude",                       DoubleType(),    True),
  StructField("longitude",                      DoubleType(),    True),
  StructField("x_coordinate",                   DoubleType(),    True),
  StructField("y_coordinate",                   DoubleType(),    True),
  StructField("record_version",                 TimestampType(), True),
  StructField("batch_id",                       StringType(),    True),
  StructField("source_file",                    StringType(),    True),
  StructField("load_timestamp",                 TimestampType(), False),
])

print(len(silver_schema.fields), "silver columns")

## 3. Data dictionary

Written to a table so it can be queried, and printed as markdown for the
README. The primary key of Silver is `unique_key` — one row per service
request, unlike Bronze which keeps every version received.

In [ ]:
SILVER_DOCS = {
  "unique_key": "Unique identifier of the service request (PRIMARY KEY)",
  "created_date": "When the request was created",
  "closed_date": "When the request was closed; null while still open",
  "due_date": "Date the agency is expected to resolve the request by",
  "resolution_action_updated_date": "When the resolution was last updated",
  "agency": "Acronym of the responding city agency",
  "agency_name": "Full name of the responding city agency",
  "complaint_type": "Primary category of the complaint",
  "descriptor": "Detail describing the complaint type",
  "location_type": "Type of location, e.g. street or residential building",
  "status": "Assigned, Pending, Closed or Canceled",
  "resolution_description": "Free text describing how the request was resolved",
  "incident_zip": "ZIP code of the incident",
  "city": "City of the incident",
  "borough": "Borough of the incident",
  "community_board": "Community board responsible for the area",
  "council_district": "City council district number",
  "police_precinct": "NYPD precinct covering the incident",
  "open_data_channel_type": "How the request was reported: phone, online, mobile, other",
  "address_type": "How the location was specified: address, intersection, block face",
  "latitude": "Latitude of the incident",
  "longitude": "Longitude of the incident",
  "x_coordinate": "State plane X coordinate",
  "y_coordinate": "State plane Y coordinate",
  "record_version": "Recency key used to pick the newest version of a request",
  "batch_id": "Batch the surviving version came from (lineage)",
  "source_file": "Raw file the surviving version came from (lineage)",
  "load_timestamp": "When this record was processed into Silver",
}

data_dictionary = spark.createDataFrame(
    [(f.name,
      f.dataType.simpleString(),
      "YES" if f.nullable else "NO",
      "PK" if f.name == "unique_key" else "",
      SILVER_DOCS[f.name])
     for f in silver_schema.fields],
    ["column_name", "data_type", "nullable", "key", "description"])

data_dictionary.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.silver_data_dictionary")

display(data_dictionary)

In [ ]:
# Markdown version for the README
print("| Column | Type | Nullable | Key | Description |")
print("|---|---|---|---|---|")
for r in data_dictionary.collect():
    print(f"| {r.column_name} | {r.data_type} | {r.nullable} | "
          f"{r.key} | {r.description} |")

## 4. Execution log

In [ ]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    layer STRING, parameter STRING,
    start_time TIMESTAMP, end_time TIMESTAMP, status STRING,
    rows_read BIGINT, rows_inserted BIGINT, rows_updated BIGINT,
    error_message STRING) USING DELTA""")

def write_log(parameter, start, status, rows_read=None,
              rows_inserted=None, rows_updated=None, error=None):
    spark.sql(f"""INSERT INTO {LOG_TABLE}
        (layer, parameter, start_time, end_time, status,
         rows_read, rows_inserted, rows_updated, error_message)
        VALUES (
        'Bronze-to-Silver', '{parameter}',
        TIMESTAMP '{start}', current_timestamp(), '{status}',
        {rows_read     if rows_read     is not None else 'NULL'},
        {rows_inserted if rows_inserted is not None else 'NULL'},
        {rows_updated  if rows_updated  is not None else 'NULL'},
        {repr(str(error)[:500]) if error else 'NULL'})""")
    print(f"[log] Bronze-to-Silver | {parameter} | {status}")

## 5. Full rebuild

Only runs when `silver_mode = full_rebuild`.

In [ ]:
if silver_mode == "full_rebuild":
    spark.sql(f"DROP TABLE IF EXISTS {SILVER}")
    print("Silver dropped - will be rebuilt from Bronze")

## 6. Read from Bronze

`batch` reads one batch, the other two modes read everything.

In [ ]:
start_time = datetime.now()

src = spark.table(BRONZE)
if silver_mode == "batch":
    src = src.filter(col("batch_id") == batch_id)

rows_read = src.count()
print("rows read from bronze:", rows_read)

if rows_read == 0:
    raise ValueError(f"No bronze rows found for batch_id = {batch_id}")

## 7. Cast to the contract

Bronze holds everything as text or loose types. Here each column is cast to its
declared Silver type. Spark returns null when a cast fails, so a value that was
present in Bronze but null after casting is a type violation, which is exactly
what the quarantine in the next cell catches.

In [ ]:
typed = (src
    .withColumn("created_date",     col("created_date").cast("timestamp"))
    .withColumn("closed_date",      col("closed_date").cast("timestamp"))
    .withColumn("due_date",         col("due_date").cast("timestamp"))
    .withColumn("resolution_action_updated_date",
                col("resolution_action_updated_date").cast("timestamp"))
    .withColumn("council_district", col("council_district").cast("integer"))
    .withColumn("latitude",         col("latitude").cast("double"))
    .withColumn("longitude",        col("longitude").cast("double"))
    .withColumn("x_coordinate",     col("x_coordinate").cast("double"))
    .withColumn("y_coordinate",     col("y_coordinate").cast("double"))
    .withColumn("agency",           trim(col("agency")))
    .withColumn("agency_name",      trim(col("agency_name")))
    .withColumn("complaint_type",   trim(col("complaint_type")))
    .withColumn("descriptor",       trim(col("descriptor")))
    .withColumn("status",           trim(col("status")))
    .withColumn("borough",          trim(col("borough")))
    .withColumn("record_version",
                coalesce(col("resolution_action_updated_date"),
                         col("created_date")))
    .withColumn("load_timestamp",   current_timestamp()))

typed.write.format("delta").mode("overwrite") \
     .saveAsTable("workspace.default._tmp_silver_typed")
typed = spark.table("workspace.default._tmp_silver_typed")

print("cast complete")

## 8. Schema drift — quarantine non-conforming records

Two kinds of violation are caught here rather than crashing the batch:

- a missing `unique_key`, which the contract declares NOT NULL
- a value that was present in Bronze but became null after casting, which means
  the source changed type (for example a number arriving as text)

Everything else continues through.

In [ ]:
CAST_CHECKS = [("created_date",     "timestamp"),
               ("council_district", "integer"),
               ("latitude",         "double"),
               ("longitude",        "double")]

bad_cast = None
for name, _ in CAST_CHECKS:
    cond = src.select("unique_key", col(name).alias("raw")) \
              .join(typed.select("unique_key", col(name).alias("cast")),
                    "unique_key")
    cond = cond.filter(col("raw").isNotNull() & col("cast").isNull()) \
               .withColumn("reason", lit(f"{name}: cast to its declared type failed"))
    bad_cast = cond.select("unique_key", "reason") if bad_cast is None \
               else bad_cast.union(cond.select("unique_key", "reason"))

bad_key = (typed.filter(col("unique_key").isNull())
                .select(lit(None).cast("string").alias("unique_key"),
                        lit("unique_key is null (violates NOT NULL)").alias("reason")))

quarantine_df = (bad_cast.union(bad_key)
                 .withColumn("batch_id", lit(batch_id if silver_mode == "batch"
                                             else silver_mode))
                 .withColumn("load_timestamp", current_timestamp()))

bad_count = quarantine_df.count()

if bad_count > 0:
    print("quarantining", bad_count, "non-conforming records")
    quarantine_df.limit(0).write.format("delta") \
        .mode("ignore").saveAsTable(QUARANTINE)
    quarantine_df.write.format("delta").mode("append").saveAsTable(QUARANTINE)
else:
    print("no schema violations")

## 9. One row per request

A request appears in Bronze once per batch it arrived in. The window keeps the
version with the latest `record_version`, so a closed request is never
overwritten by its earlier open version. `dropDuplicates` would keep an
arbitrary row and silently lose status updates.

In [ ]:
newest = Window.partitionBy("unique_key").orderBy(
    col("record_version").desc_nulls_last())

silver_df = (typed
    .filter(col("unique_key").isNotNull())
    .withColumn("_rn", row_number().over(newest))
    .filter(col("_rn") == 1).drop("_rn")
    .select([col(f.name).cast(f.dataType) for f in silver_schema.fields]))

print("rows after dedup:", silver_df.count())

## 10. MERGE INTO

Upsert on the primary key. Running this twice inserts nothing the second time,
which is what makes the pipeline idempotent. The update is conditional on the
incoming version being newer, so replaying an old batch cannot overwrite
fresher data.

In [ ]:
try:
    silver_df.limit(0).write.format("delta").mode("ignore").saveAsTable(SILVER)

    (DeltaTable.forName(spark, SILVER).alias("target")
        .merge(silver_df.alias("source"),
               "target.unique_key = source.unique_key")
        .whenMatchedUpdateAll(
            condition="source.record_version > target.record_version "
                      "OR target.record_version IS NULL")
        .whenNotMatchedInsertAll()
        .execute())

    metrics = spark.sql(f"DESCRIBE HISTORY {SILVER} LIMIT 1") \
                   .select("operationMetrics").collect()[0][0] or {}
    inserted = int(metrics.get("numTargetRowsInserted", 0))
    updated  = int(metrics.get("numTargetRowsUpdated", 0))

    write_log(batch_id if silver_mode == "batch" else silver_mode,
              start_time, "SUCCESS", rows_read, inserted, updated)
    print("inserted:", inserted, "| updated:", updated)

except Exception as e:
    write_log(batch_id if silver_mode == "batch" else silver_mode,
              start_time, "FAILURE", rows_read, error=e)
    raise

spark.sql("DROP TABLE IF EXISTS workspace.default._tmp_silver_typed")

## 11. Idempotency test

Re-running the same MERGE must not change the row count and must leave zero
duplicate keys.

In [ ]:
before = spark.table(SILVER).count()

(DeltaTable.forName(spark, SILVER).alias("target")
    .merge(silver_df.alias("source"), "target.unique_key = source.unique_key")
    .whenMatchedUpdateAll(
        condition="source.record_version > target.record_version "
                  "OR target.record_version IS NULL")
    .whenNotMatchedInsertAll()
    .execute())

after = spark.table(SILVER).count()
dups  = spark.table(SILVER).groupBy("unique_key").count() \
             .filter("count > 1").count()

print("before:", before, "| after:", after, "| duplicate keys:", dups)
assert before == after and dups == 0, "IDEMPOTENCY FAILED"
print("Idempotent.")

## 12. Verify

In [ ]:
print("silver rows:", spark.table(SILVER).count())

spark.table(SILVER).select(
    "unique_key", "created_date", "closed_date", "complaint_type",
    "borough", "status", "batch_id", "load_timestamp").show(5, truncate=False)

display(spark.table(LOG_TABLE).orderBy(col("start_time").desc()))